In [ ]:
import sys
sys.path.insert(0, '../frameaxis')
from frameaxis import CoreUtil
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import pickle

In [17]:
news_df = pd.read_csv(r'C:\Users\fuent\uchile\semestre-10\memoria\frameaxis\chilean-news\dataset\news_sample_with_summaries.csv', dtype={'uri': str})

In [18]:
frameaxis = pickle.load(open('frameaxis_object.p', 'rb'))

## Función para obtener el artículo con mayor variación en el sesgo e intensidad

In [41]:
def top_n_microframe_var(
    tsv_body_path, 
    tsv_summary_path,
    reverse=False,
    n=10
):
    # Cargar los archivos TSV
    df_body = pd.read_csv(tsv_body_path, sep="\t", dtype={'uri': str})
    df_summary = pd.read_csv(tsv_summary_path, sep="\t", dtype={'uri': str})

    # Merge directo por 'uri'
    merged = pd.merge(df_body, df_summary, on="uri", suffixes=("_body", "_summary"))
    microframes = [col for col in df_body.columns if col != "uri"]

    # Vectorizado: calcular diferencias para todas las columnas a la vez
    body_cols = [f"{mf}_body" for mf in microframes]
    summary_cols = [f"{mf}_summary" for mf in microframes]
    diff = merged[summary_cols].values - merged[body_cols].values
    if 'intensity' in tsv_body_path:
        # Evitar división por cero
        with np.errstate(divide='ignore', invalid='ignore'):
            diff = np.where(merged[body_cols].values != 0, diff / merged[body_cols].values, np.nan)

    # Reestructurar a formato largo (long-form) usando numpy y pandas
    n_rows = merged.shape[0]
    data = {
        'uri': np.repeat(merged['uri'].values, len(microframes)),
        'microframe': np.tile(microframes, n_rows),
        'diff': diff.flatten(),
        'body': merged[body_cols].values.flatten(),
        'summary': merged[summary_cols].values.flatten()
    }
    df_long = pd.DataFrame(data)
    
    # Ordenar y devolver top n
    top = df_long.sort_values('diff', ascending=not reverse).head(n)
    return top.reset_index(drop=True)

In [33]:
def top_word_contributions_diff(top_microframe_df, metric, ascending=False, n=10):

    # Obtener todos los textos de la columna 'body' y 'summary' como listas de strings
    body = news_df[news_df['uri'] == top_microframe_df['uri'].values[0]]['body'].values[0]
    summary = news_df[news_df['uri'] == top_microframe_df['uri'].values[0]]['summary'].values[0]
    axis = top_microframe_df['microframe'].values[0]

    # Preprocesar cada texto en ambas listas
    preprocessed_body = CoreUtil.preprocess(body)
    preprocessed_summary = CoreUtil.preprocess(summary)

    # Calcular las contribuciones para los artículos originales y sus resumenes
    if metric == 'bias':
        contrib_body, words_body = frameaxis.word_contribution(
            [preprocessed_body], 
            ast.literal_eval(axis),
            min_freq=1
        )
        contrib_summary, words_summary = frameaxis.word_contribution(
            [preprocessed_summary], 
            ast.literal_eval(axis),
            min_freq=1
        )
    elif metric == 'intensity':
        df_corpus_model = pd.read_csv(f"results/bias_table_gpt_summaries.tsv", sep="\t", dtype={'uri': str})
        df_corpus_body = pd.read_csv(f"results/bias_table_body.tsv", sep="\t", dtype={'uri': str})

        # para el microframe a usar promedia el sesgo a lo largo de los textos
        corpus_model_mean = np.mean(df_corpus_model[axis])
        corpus_body_mean = np.mean(df_corpus_body[axis])

        contrib_body, words_body = frameaxis.word_contribution_to_second_moment(
            [preprocessed_body], 
            ast.literal_eval(axis),
            corpus_body_mean,
            min_freq=1
        )
        contrib_summary, words_summary = frameaxis.word_contribution_to_second_moment(
            [preprocessed_summary], 
            ast.literal_eval(axis),
            corpus_model_mean,
            min_freq=1
        )

    # Crear DataFrames con las contribuciones
    df_body = pd.DataFrame({
        'word': words_body,
        'contribution_body': contrib_body[0][0]
    })

    df_summary = pd.DataFrame({
        'word': words_summary,
        'contribution_summary': contrib_summary[0][0]
    })

    # Obtener todas las palabras únicas de ambas listas
    all_words = set(words_body).union(set(words_summary))

    # Asegurarse de que todas las palabras estén presentes en ambos DataFrames
    df_body = df_body.set_index('word').reindex(all_words, fill_value=0).reset_index()
    df_summary = df_summary.set_index('word').reindex(all_words, fill_value=0).reset_index()

    # Unir ambos DataFrames por la palabra
    merged_df = pd.merge(df_body, df_summary, on='word', how='inner')

    # Calcular la variación en la contribución
    merged_df['variation'] = merged_df['contribution_summary'] - merged_df['contribution_body']

    # Agregar la variación absoluta y obtener las top n variaciones
    merged_df['abs_variation'] = merged_df['variation'].abs()

    top_variations = merged_df.sort_values('abs_variation', ascending=ascending).reset_index(drop=True).head(n)

    return top_variations

## Artículo con mayor variación en el sesgo

In [34]:
gpt_bias_pos = top_n_microframe_var(
    r'C:\Users\fuent\uchile\semestre-10\memoria\frameaxis\chilean-news\results\bias_table_body.tsv',
    r'C:\Users\fuent\uchile\semestre-10\memoria\frameaxis\chilean-news\results\bias_table_gpt_summaries.tsv',
    reverse=True,
    n=1
)

In [35]:
gpt_bias_pos

,uri,microframe,diff,body,summary
0,7159524639,"('contento', 'descontento')",0.108225,-0.064975,0.043251


In [36]:
news_df[news_df['uri'] == gpt_bias_pos['uri'].values[0]]

,uri,title,body,outlet,summary,summary_claude,summary_gemini
826,7159524639,"""¡Tan matando a un hueón!"": Zalo Reyes y un te...","-- El Zalo Reyes nunca me ha querido a mí, por...",La Cuarta,"En el programa ""Cara & Sello"", Zalo Reyes y su...",\n\nEl texto narra un encontronazo entre el c...,El resumen del texto es el siguiente:\n\nEl ar...


In [30]:
print(news_df[news_df['uri'] == gpt_bias_pos['uri'].values[0]]['body'].values[0])

-- El Zalo Reyes nunca me ha querido a mí, porque dice que yo le hago sombra.

Lo dice un hombre físicamente parecido al "Gorrión de Conchalí", al comienzo de un recordado capítulo del programa de Mega, Cara & Sello, titulado simplemente "Zalo Reyes y su doble".

Quien habla es Carlos Caro, un sujeto que por entonces imitaba hace más de dos décadas a la voz de "Historia de un amor" y "Mi prisionera".

En el polémico programa de Mega ambos cantantes contaron sus vidas y se enfrascaron en una comentada escalada de dimes y diretes.

El encontrón acabaría con uno de los protagonistas expulsado de una famosa parrillada santiaguina.

Incluso, varios años después, Felipe Avello utilizó una frase del encuentro en sus rutinas en los festivales de Olmué y Viña del Mar.

En el segmento final de Cara & sello, luego que mostraron cómo se desenvuelven artísticamente Zalo Reyes y su imitador, el Gorrión contó que había tenido un problema previo con su doble:

-- Y si anda subversivo y con ganas de...

In [32]:
print(news_df[news_df['uri'] == gpt_bias_pos['uri'].values[0]]['summary'].values[0])

En el programa "Cara & Sello", Zalo Reyes y su imitador, Carlos Caro, protagonizan un intenso intercambio sobre su relación, marcada por la rivalidad y malentendidos. Zalo expresa que Caro le hace sombra y que no siente interés en mantener contacto, mientras que Caro defiende su talento y reclama respeto como artista. Un conflicto previo en un restaurante se menciona, y ambos discuten sobre una torta que Caro llevó a la casa de Zalo, quien se muestra reacio a socializar debido a problemas personales. La situación escaló en tensiones y malentendidos, culminando en un abrupto final donde Zalo se siente humillado y Caro defiende su dignidad como artista de barrio. Ambos expresan sus sentimientos y frustraciones, mostrando el lado humano detrás de la rivalidad.


In [28]:
top_word_contributions_diff(gpt_bias_pos, 'bias', ascending=False, n=10)

c:\Users\fuent\AppData\Local\pypoetry\Cache\virtualenvs\frameaxis-nVQu9Ozg-py3.8\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


,word,contribution_body,contribution_summary,variation
0,rivalidad,0.000000,0.007129,0.007129
1,malentendidos,0.000000,0.006216,0.006216
2,tensiones,0.000000,0.004919,0.004919
3,conflicto,0.000000,0.004446,0.004446
4,hueón,-0.003982,0.000000,0.003982
5,frustraciones,0.000000,0.003894,0.003894
6,si,-0.003632,0.000000,0.003632
7,interés,0.000000,0.002973,0.002973
8,bien,-0.002544,0.000000,0.002544
9,debido,0.000000,0.002536,0.002536


## Artículo con mayor variación en la intensidad

In [42]:
gpt_intensity = top_n_microframe_var(
    r'C:\Users\fuent\uchile\semestre-10\memoria\frameaxis\chilean-news\results\intensity_table_body.tsv',
    r'C:\Users\fuent\uchile\semestre-10\memoria\frameaxis\chilean-news\results\intensity_table_gpt_summaries.tsv',
    reverse=True,
    n=1
)

In [43]:
gpt_intensity

,uri,microframe,diff,body,summary
0,6897337010,"('negativo', 'neutral')",7.354932,0.001576,0.013168


In [44]:
news_df[news_df['uri'] == gpt_intensity['uri'].values[0]]

,uri,title,body,outlet,summary,summary_claude,summary_gemini
1129,6897337010,"Delgado a Siches sobre Interior: ""Es un privil...",El ministro del Interior y la nueva jefa de ga...,El Mercurio de Santiago,"El ministro del Interior, Rodrigo Delgado, se ...","El ministro del Interior, Rodrigo Delgado, se...","El ministro del Interior actual, Rodrigo Delga..."


In [45]:
print(news_df[news_df['uri'] == gpt_intensity['uri'].values[0]]['body'].values[0])

El ministro del Interior y la nueva jefa de gabinete en una reunión anterior en La Moneda

El ministro del Interior, Rodrigo Delgado, contó que se contactó ayer con la nominada en su mismo cargo por el Presidente electo, Gabriel Boric, Izkia Siches, para felicitarla y mostrarle su disposición a reunirse para realizar un traspaso de la cartera.

Si bien la reunión protocolar está fijada para el 21 de febrero, el jefe de gabinete no descartó programar citas antes con la ex presidenta del Colegio Médico para informarle sobre algunas políticas de Estado en las que él ha estado trabajando.

Delgado además le dio algunos consejos sobre cómo es encabezar la cartera de Interior, descartando que se trate de una "moledora de carne" como dijo otro ex ministro del Interior, Jorge Burgos (DC).


In [46]:
print(news_df[news_df['uri'] == gpt_intensity['uri'].values[0]]['summary'].values[0])

El ministro del Interior, Rodrigo Delgado, se contactó con la nominada Izkia Siches para felicitarla y ofrecerle su apoyo en el traspaso de la cartera, programado para el 21 de febrero. Además, Delgado sugirió organizar reuniones previas para informarle sobre políticas estatales en las que ha estado trabajando y compartió consejos sobre cómo liderar el Ministerio, aclarando que no es tan negativo como lo describió un ex ministro.


In [47]:
top_word_contributions_diff(gpt_intensity, 'intensity', ascending=False, n=10)

c:\Users\fuent\AppData\Local\pypoetry\Cache\virtualenvs\frameaxis-nVQu9Ozg-py3.8\lib\site-packages\sklearn\utils\deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


,word,contribution_body,contribution_summary,variation,abs_variation
0,negativo,0.000000,0.010938,0.010938,0.010938
1,organizar,0.000000,0.000466,0.000466,0.000466
2,ofrecerle,0.000000,0.000373,0.000373,0.000373
3,liderar,0.000000,0.000258,0.000258,0.000258
4,jorge,0.000133,0.000000,-0.000133,0.000133
5,carne,0.000111,0.000000,-0.000111,0.000111
6,sugirió,0.000000,0.000111,0.000111,0.000111
7,reunirse,0.000102,0.000000,-0.000102,0.000102
8,nueva,0.000099,0.000000,-0.000099,0.000099
9,descartando,0.000066,0.000000,-0.000066,0.000066
